In [ ]:
import os
import io
import cv2
import time
import math
import numpy as np
import pandas as pd
import albumentations as A
import matplotlib.pyplot as plt
from PIL import Image

In [ ]:
work_dir = "/kaggle/working/"
data_dir = "../input/understanding_cloud_organization"
train_csv_path = os.path.join(data_dir,'train.csv')
test_csv_path = os.path.join(data_dir,"sample_submission.csv")
train_image_path = os.path.join(data_dir,'train_images')
test_image_path = os.path.join(data_dir,'test_images')

In [ ]:
class COLOR:
    PURPLE = '\033[95m'
    CYAN = '\033[96m'
    DARKCYAN = '\033[36m'
    BLUE = '\033[94m'
    GREEN = '\033[92m'
    YELLOW = '\033[93m'
    RED = '\033[91m'
    BOLD = '\033[1m'
    UNDERLINE = '\033[4m'
    END = '\033[0m'

# Reading and Preaparing the Dataframe

`Most of the Idea of this section has been taken from this notebook` - https://www.kaggle.com/code/ekhtiar/tf-tutorial-semantic-segmentation-with-u-net

In [ ]:
train_df = pd.read_csv(train_csv_path).fillna(-1)
train_df.head()

In [ ]:
train_df['Image_Id'] = train_df['Image_Label'].apply(lambda x: x.split('_')[0])
train_df['Label'] = train_df['Image_Label'].apply(lambda x: x.split('_')[1])
train_df.head()

In [ ]:
train_df['Label_EncodedPixels'] = train_df.apply(lambda row: (row['Label'], row['EncodedPixels']), axis = 1)
train_df.head()

In [ ]:
grouped_EncodedPixels = train_df.groupby('Image_Id')['Label_EncodedPixels'].apply(list)
grouped_EncodedPixels.head()
grouped_EncodedPixels.info()

In [ ]:
train_df = grouped_EncodedPixels.to_frame().reset_index()
train_df.head()

In [ ]:
labels = ['Fish', 'Flower', 'Gravel', 'Sugar']

for label in labels:
    train_df = train_df.assign(**{label: 0})
for index, row in train_df.iterrows():
    for item in row['Label_EncodedPixels']:
        label, value = item
        if value == -1:
            bool_value = 0
        else:
            bool_value = 1
        train_df.loc[index, label] = bool_value

train_df['classes'] = train_df.apply(lambda row: [col for col in labels if row[col] == 1], axis=1)

train_df.head()

In [ ]:
train_df.info()

In [ ]:
# printing out indexes (in dataframe) of some images having all 4 types of mask
for ix,item in enumerate(train_df['Label_EncodedPixels'][:100]):
    c1=item[0][-1]!=-1
    c2=item[1][-1]!=-1
    c3=item[2][-1]!=-1
    c4=item[3][-1]!=-1
    if c1 and c2 and c3 and c4:
        print(ix) 

In [ ]:
train_df.loc[18]["Image_Id"]

In [ ]:
# printing out indexes (in image dir) of some images having all 4 types of mask
for ix,item in enumerate(os.listdir(train_image_path)):
    if item == train_df.loc[18]["Image_Id"]:
        print(ix)

In [ ]:
for j,item in enumerate(train_df['Label_EncodedPixels'][:100]):
    c1=item[0][-1]!=-1
    c2=item[1][-1]!=-1
    c3=item[2][-1]!=-1
    c4=item[3][-1]!=-1
    if c1 and c2 and c3 and c4:
        for ix,item in enumerate(os.listdir(train_image_path)):
            if item == train_df.loc[j]["Image_Id"]:
                print(ix)

In [ ]:
INDEX = 5390

In [ ]:
indexes =  [0, 1, 2, 3]
labels = ['Fish', 'Flower', 'Gravel', 'Sugar']
colors = ['maroon', 'darkblue', 'purple','teal']
colormaps = ['PuRd_r', 'Blues_r', 'Purples_r','winter_r']
rgb_colors = [(56, 255, 255),(255, 70, 90),(48, 255, 99),(255, 255, 102)]

label_to_idx = dict(zip(labels,indexes))
idx_to_label = dict(zip(indexes,labels))

label_to_color = dict(zip(labels,colors))
idx_to_color = dict(zip(indexes,colors))

label_to_rgb_color =  dict(zip(labels,rgb_colors))
idx_to_rgb_color = dict(zip(indexes,rgb_colors))

label_to_colormap = dict(zip(labels,colormaps))
idx_to_colormap = dict(zip(indexes,colormaps))

# Exploratory Data Analysis

In [ ]:
def count_nonzero_pixel_from_rle(rle_string, height, width):
    """
    Counts non-zero pixels in a RLE-encoded mask without decoding.

    Args:
        rle_string (str): RLE-encoded mask string.
        height (int): Height of the mask.
        width (int): Width of the mask.

    Returns:
        int: Number of non-zero pixels in the mask.
    """

    if rle_string == -1:
        return 0  # Empty mask

    nonzero_count = 0
    rle_numbers = [int(num_string) for num_string in rle_string.split(' ')]
    rle_pairs = np.array(rle_numbers).reshape(-1, 2)

    for index, length in rle_pairs:
        if length > 0:  # Only count non-zero runs
            nonzero_count += length

    return nonzero_count

## Classwise Pixel Count

In [ ]:
%%time 

class_wise_pixel_count = {
    "Fish":0,
    "Flower":0,
    "Gravel":0,
    "Sugar":0
}

img_width = 2100
img_height = 1400

fish_count = 0
flower_count = 0 
gravel_count = 0
sugar_count = 0

for ix, item in train_df.iterrows():
    rle = item["Label_EncodedPixels"]
    class_wise_pixel_count["Fish"] += count_nonzero_pixel_from_rle(rle[0][1], img_height, img_width)
    class_wise_pixel_count["Flower"] += count_nonzero_pixel_from_rle(rle[1][1], img_height, img_width)
    class_wise_pixel_count["Gravel"] += count_nonzero_pixel_from_rle(rle[2][1], img_height, img_width)
    class_wise_pixel_count["Sugar"] += count_nonzero_pixel_from_rle(rle[3][1], img_height, img_width)
    

print(class_wise_pixel_count)

In [ ]:
fig = plt.figure(num=None, figsize=(18, 6), dpi=80, facecolor='w', edgecolor='k')
fig.suptitle('Classwise Pixel Count', fontsize=20)
fig.tight_layout();

ax = plt.subplot(1,2,1)
bar = plt.bar(class_wise_pixel_count.keys(), class_wise_pixel_count.values(), color=colors);
for rect in bar:
    height = rect.get_height()
    plt.text(rect.get_x() + rect.get_width()/2, height, '%.3E' % height,
             ha='center', va='bottom',fontsize=10)
plt.xlabel("Cloud Types");
plt.ylabel("Total Pixels");

ax = plt.subplot(1,2,2)
plt.pie(class_wise_pixel_count.values(),
        labels=class_wise_pixel_count.keys(),
        autopct='%1.1f%%',
        explode=[0.1,0,0,0],
        colors=colors,
        shadow=True, 
        startangle=0);

print(COLOR.BOLD +COLOR.GREEN+ "Observation: The pixel distribution of the classes is somewhat balanced." + COLOR.END)

## Pixel Distribution

In [ ]:
total_pixels = img_height*img_width*len(train_df)
mask_pixels = sum(class_wise_pixel_count.values())

pixel_distribution = {"Background":(total_pixels-mask_pixels)/total_pixels*100,
                       "Fish":class_wise_pixel_count["Fish"]/total_pixels*100,
                       "Flower":class_wise_pixel_count["Flower"]/total_pixels*100,
                       "Gravel":class_wise_pixel_count["Gravel"]/total_pixels*100,
                       "Sugar":class_wise_pixel_count["Sugar"]/total_pixels*100,
                       }

print(pixel_distribution)

In [ ]:
fig = plt.figure(num=None, figsize=(18, 6), dpi=80, facecolor='w', edgecolor='k')
fig.suptitle('Pixel Distribution', fontsize=24)
fig.tight_layout();

ax = plt.subplot(1,2,1)
bar = plt.bar(pixel_distribution.keys(), pixel_distribution.values(), color=["darkslategrey"]+colors);
for rect in bar:
    height = rect.get_height()
    plt.text(rect.get_x() + rect.get_width()/2, height, '%.3f %%' % height,
             ha='center', va='bottom',fontsize=10)
    
plt.xlabel("Pixel Types");
plt.ylabel("Parcentage of Pixels");

ax = plt.subplot(1,2,2)
plt.pie(pixel_distribution.values(),
        labels=pixel_distribution.keys(),
        autopct='%1.1f%%',
        explode=[0.1,0,0,0,0],
        shadow=True,
        colors=["darkslategrey"]+colors,
        startangle=180);


comment = '''Observation: The pixel distribution of the overall dataset is not balanced,
             most of the pixels does not have any associated label.'''

print(COLOR.BOLD +COLOR.GREEN+ comment + COLOR.END)

## Class Frequency

In [ ]:
fig = plt.figure(num=None, figsize=(12, 6), dpi=80, facecolor='w', edgecolor='k')
fig.tight_layout();

class_frequency = dict(train_df[labels].sum())
bar = plt.bar(class_frequency.keys(),class_frequency.values(), color=colors)
for rect in bar:
    height = rect.get_height()
    plt.text(rect.get_x() + rect.get_width()/2, height, height,
             ha='center', va='bottom',fontsize=10)
    
plt.xlabel("Labels",fontsize=16);
plt.ylabel('Frequency',fontsize=16)
plt.title("Mask Class Frequency",fontsize=20)

print(COLOR.BOLD +COLOR.GREEN+ "Observation: The Dataset is somewhat balanced for classifcation tasks." + COLOR.END)

## Class Frequency per Image

In [ ]:
fig = plt.figure(num=None, figsize=(12, 6), dpi=80, facecolor='w', edgecolor='k')
fig.tight_layout();

class_frequency_per_image = dict(train_df["classes"].apply(len).value_counts())

bar = plt.bar(class_frequency_per_image.keys(), class_frequency_per_image.values(), color=colors);
for rect in bar:
    height = rect.get_height()
    plt.text(rect.get_x() + rect.get_width()/2, height, height,
             ha='center', va='bottom',fontsize=10)
plt.xlabel("No. of Labels in a Single Image",fontsize=16);
plt.xticks(ticks=[1,2,3,4]);
plt.ylabel("Frequency",fontsize=16);
plt.title("Class Frequency per Image",fontsize=20);

print(COLOR.BOLD +COLOR.GREEN+ "Observation: Most of the images have 2 labels." + COLOR.END)

## Average Mask Area

In [ ]:
#Avarage Area per mask
avarage_mask_area = dict()

for key in class_frequency.keys():
    avarage_mask_area[key] = class_wise_pixel_count[key]//class_frequency[key]
    
fig = plt.figure(num=None, figsize=(12, 6), dpi=80, facecolor='w', edgecolor='k')
fig.tight_layout();

bar = plt.bar(avarage_mask_area.keys(), avarage_mask_area.values(), color=colors);
for rect in bar:
    height = rect.get_height()
    plt.text(rect.get_x() + rect.get_width()/2, height, '%.3E'%height,
             ha='center', va='bottom',fontsize=10)
    
plt.xlabel("Labels",fontsize=16);
plt.ylabel("Average no. of Pixels per mask",fontsize=16);
plt.title("Average Mask Area",fontsize=20);

print(COLOR.BOLD +COLOR.GREEN+ "Observation: Avarage area(pixel count) for each mask are somewhat close." + COLOR.END)

## Class Combination Frequency

In [ ]:
from itertools import combinations
classes = labels
combinations_list = list(combinations(classes, 1)) + list(combinations(classes, 2)) + list(combinations(classes, 3)) + list(combinations(classes, 4))

label_counts = {}
for combination in combinations_list:
    count = train_df[train_df["classes"].apply(lambda x: set(combination).issubset(x))].shape[0]
    label_counts[tuple(combination)] = count
    

def remove_chars_iter(subj,):
    chars = [")","(","'"]
    subj = str(subj)
    sc = set(chars)
    return ''.join([c for c in subj if c not in sc]);

#remove_chars_iter(list(label_counts.keys())[5])

    
label_counts

In [ ]:
fig = plt.figure(num=None, figsize=(12, 6), dpi=80, facecolor='w', edgecolor='k')
fig.tight_layout();


bar = plt.barh(list(map(remove_chars_iter,label_counts.keys())), label_counts.values());

for rect in bar:
    width = rect.get_width()
    plt.text(width-width/3, rect.get_y() + rect.get_height()/5, width,
             ha='center', va='bottom',fontsize=8)
    
plt.xlabel("Frequency (No. of Images)", fontsize=16);
plt.ylabel("Class Combiantions", fontsize=16);
plt.title("Class Combination Frequency", fontsize=20);

# Visualizing the Images
`PIL or CV2 Image (image_width, image_height) == Numpy Array (image_height, image_width)`

In [ ]:
def batchDataLoader(image_dir,img_w= 512, img_h=512, num_channel =4, Batch_Size=32):
    
    while True:
        k=0
        image_ids = os.listdir(image_dir)
        num_batches = math.ceil(len(image_ids)/Batch_Size)
        
        for batch_no in range(1,num_batches+1): 
            if batch_no < num_batches:
                batch_size = Batch_Size
                batch_image_ids = image_ids[k:k+batch_size]
                image_batch = np.zeros((batch_size, img_h, img_w, num_channel),dtype=np.uint8)
                for i in range(batch_size):
                    path = os.path.join(image_dir, image_ids[i])
                    img = cv2.imread(path)
                    img =  cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                    image_batch[i] = img
            # for the last batch which could be fractional
            if batch_no == num_batches:
                batch_image_ids = image_ids[k:]
                batch_size = len(batch_image_ids)
                image_batch = np.zeros((batch_size, img_h, img_w, num_channel),dtype=np.uint8)
                for i in range(batch_size):
                    path = os.path.join(image_dir, image_ids[i])
                    img = cv2.imread(path)
                    img =  cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                    image_batch[i] = img
            
            k = k+batch_size
            print(f"batch_no = {batch_no}")
            yield image_batch

In [ ]:
img_width = 2100
img_height = 1400
num_channel = 3
BATCH_SIZE = 32
current_batch = batchDataLoader(train_image_path,img_width,img_height, num_channel, BATCH_SIZE)

In [ ]:
images = next(current_batch)
print(images.shape)
plt.figure(figsize=(24,8))
for i in range(8):
    ax = plt.subplot(2,4, i+1)
    plt.imshow(images[i])
    plt.axis("off");

# Visualizing the Segmentation Masks

`rle_to_mask function source` - https://www.kaggle.com/robertkag/rle-to-mask-converter

In [ ]:
def rle_to_mask(rle_string, height, width):
    '''
    convert RLE(run length encoding) string to numpy array

    Parameters: 
    rle_string (str): string of rle encoded mask
    height (int): height of the mask
    width (int): width of the mask 

    Returns: 
    numpy.array: numpy array of the mask
    '''
    
    rows, cols = height, width
    
    if rle_string == -1:
        return np.zeros((height,width))
    else:
        rle_numbers = [int(num_string) for num_string in rle_string.split(' ')]
        rle_pairs = np.array(rle_numbers).reshape(-1,2)
        img = np.zeros(rows*cols, dtype=np.uint8)
        for index, length in rle_pairs:
            index -= 1
            img[index:index+length] = 255
        img = img.reshape(cols,rows)
        img = img.T
        img = img/255.0
        return img

In [ ]:
image_id = '0011165.jpg'
path = os.path.join(train_image_path,image_id)
img = cv2.imread(path)
img =  cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
rle = list(train_df[train_df['Image_Id'] == image_id]['Label_EncodedPixels'])[0][0][1]

m = rle_to_mask(rle,img_height,img_width)
m = cv2.resize(m, (384,256),interpolation=cv2.INTER_LINEAR)
m = (m>0).astype(int)
plt.imshow(m)
print(m.shape)
print(np.unique(m))
print(np.argwhere(m==1)[0])

In [ ]:
def get_masks_by_img_id(dataframe, image_id):
    masks = np.zeros((img_height,img_width,4))
    rle_masks = list(dataframe[dataframe['Image_Id'] == image_id]['Label_EncodedPixels'])[0]
    fish_mask = rle_to_mask(rle_masks[0][1], img_height, img_width)
    flower_mask = rle_to_mask(rle_masks[1][1], img_height, img_width)
    gravel_mask = rle_to_mask(rle_masks[2][1], img_height, img_width)
    sugar_mask = rle_to_mask(rle_masks[3][1], img_height, img_width)
    mask_list = [fish_mask,flower_mask,gravel_mask,sugar_mask]
    for ix, mask in enumerate(mask_list):
        masks[:,:,ix] = mask
    return masks

In [ ]:
image_id = os.listdir(train_image_path)[1]
#image_id = 'f516a20.jpg'
masks = get_masks_by_img_id(train_df, image_id)
masks.shape

In [ ]:
image_id = os.listdir(train_image_path)[INDEX]
#image_id = 'f516a20.jpg'
masks = get_masks_by_img_id(train_df, image_id)
print(image_id)
plt.figure(figsize=(24,4))
for ix in range(masks.shape[-1]):
    ax = plt.subplot(1,4, ix+1)
    plt.imshow(masks[:,:,ix],cmap=None)
    plt.axis("off");

# Visualizing the Images with Segmentation Masks

In [ ]:
from matplotlib import font_manager
font_prop = font_manager.FontProperties(size=16,weight="semibold",stretch="condensed")

def draw_label_on_mask(mask, label, obj=plt):
    '''
    Function to add labels to the image.
    '''
    if np.sum(mask) > 0:
        y,x = 0,0
        y,x = np.argwhere(mask==1)[0]
        y,x = y+50,x+20      
        obj.text(x,y,label,color='white',fontproperties=font_prop)
    return None

In [ ]:
image_id = os.listdir(train_image_path)[INDEX]
path = os.path.join(train_image_path,image_id)
img = cv2.imread(path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
#img = cv2.resize(img,(384,256))
img = img.astype(np.float32)
img = img/255.0
dpi = 100
#plt.figure(figsize=(img_width/dpi, img_height/dpi), dpi=dpi)
#img -= img.mean()
#img /= img.std()
#standarization changes the color
#print(img.shape)
plt.imshow(img);
plt.axis('off');
#plt.savefig(f"{work_dir}clouds.png",transparent=True,bbox_inches='tight', pad_inches=0)

In [ ]:
image_id = os.listdir(train_image_path)[INDEX]
path = os.path.join(train_image_path,image_id)
img = cv2.imread(path)
img =  cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
masks = get_masks_by_img_id(train_df, image_id)
label = "Flower"
mask = masks[:,:,label_to_idx[label]]
mask = np.clip(mask,0,1)
mask = np.ma.masked_where(mask == 0, mask)
dpi = 100
#plt.figure(figsize=(img_width/dpi, img_height/dpi), dpi=dpi)
plt.imshow(img)
plt.imshow(mask,alpha=0.7,cmap=label_to_colormap[label])
draw_label_on_mask(mask,label)
plt.axis('off');
#plt.savefig(f"{work_dir}{label}_mask.png",transparent=True,bbox_inches='tight', pad_inches=0)

In [ ]:
image_id = os.listdir(train_image_path)[INDEX]
masks = get_masks_by_img_id(train_df, image_id)
masks = (masks[:,:,0], masks[:,:,1],masks[:,:,2],masks[:,:,3])

path = os.path.join(train_image_path,image_id)
img = cv2.imread(path)
img =  cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
colormaps = ['PuRd_r', 'Blues_r', 'Purples_r','winter_r'] # colormap_r = inverse colormap
mask_labels = ['Fish', 'Flower', 'Gravel', 'Sugar']
plt.figure(figsize=(15,10))
for i,(mask,cmap,label)in enumerate(zip(masks,colormaps,mask_labels)):
    mask = np.clip(mask,0,1)
    mask = np.ma.masked_where(mask == 0, mask)
    ax = plt.subplot(2,2, i+1)
    plt.imshow(img)
    plt.imshow(mask,alpha=0.7,cmap=cmap)
    draw_label_on_mask(mask,label)
    plt.axis("off")
    #cv2.imwrite(f"{label}.jpg",cv2.cvtColor(img, cv2.COLOR_RGB2BGR))

In [ ]:
image_id = os.listdir(train_image_path)[INDEX]
masks = get_masks_by_img_id(train_df, image_id)
masks = (masks[:,:,0], masks[:,:,1],masks[:,:,2],masks[:,:,3])

path = os.path.join(train_image_path,image_id)
img = cv2.imread(path)
img =  cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

colormaps = ['PuRd_r', 'Blues_r', 'Purples_r','winter_r']
mask_labels = ['Fish', 'Flower', 'Gravel', 'Sugar']
dpi = 100
plt.figure(figsize=(img_width/dpi, img_height/dpi), dpi=dpi)
plt.imshow(img)
for i,(mask,cmap,label) in enumerate(zip(masks,colormaps,mask_labels)):
    mask = np.clip(mask,0,1)
    mask = np.ma.masked_where(mask == 0, mask)
    plt.imshow(mask,alpha=0.7,cmap=cmap) # colormap_r = inverse colormap
    draw_label_on_mask(mask,label)
    plt.axis("off")
#plt.savefig(f"{work_dir}cloud_masks.png",transparent=True,bbox_inches='tight', pad_inches=0)

In [ ]:
image_id = os.listdir(train_image_path)[INDEX]
path = os.path.join(train_image_path,image_id)
img = cv2.imread(path)
img =  cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
masks = get_masks_by_img_id(train_df, image_id)
label = "Sugar"
mask = masks[:,:,label_to_idx[label]]
mask = np.clip(mask,0,1)
mask = np.ma.masked_where(mask == 0, mask)
bbox = cv2.boundingRect(mask.astype(np.uint8))
print(bbox)
bbox = (30, 1050, 2040, 320)
cv2.rectangle(img, bbox, label_to_rgb_color[label], 10)
cv2.putText(img, label, (bbox[0], bbox[1] + 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 2.0, (255,255,255), 6)
#dpi = 100
#plt.figure(figsize=(img_width/dpi, img_height/dpi), dpi=dpi)
plt.imshow(img)
#draw_label_on_mask(mask,label)
plt.axis('off');
#plt.savefig(f"{work_dir}{label}_cloud_bbox.png",transparent=True,bbox_inches='tight', pad_inches=0)

In [ ]:
import cv2
import numpy as np
import matplotlib

def show_bounding_boxes(image, mask, labels, colors):
    """Shows the bounding boxes surrounding the polygon in the image, and
    adds labels to the bounding boxes.

    Args:
    image: The image.
    mask: The binary polygon mask.
    labels: The labels of the objects in the mask.
    colors: A list of colors to use for the bounding boxes.

    Returns:
    The image with the bounding boxes and labels drawn on it.
    """

    # Find the bounding boxes of the polygon.
    bounding_boxes = []
    for i in range(mask.shape[-1]):
        bbox = cv2.boundingRect(mask[:, :, i])
        bounding_boxes.append(bbox)

    # Draw the bounding boxes on the image.
    for bbox, label, color_name in zip(bounding_boxes, labels, colors):
        #rgb_color = matplotlib.colors.to_rgb(color_name)
        #rgb_color = tuple(value * 255 for value in rgb_color)
        rgb_color= color_name
        cv2.rectangle(image, bbox, rgb_color, 7)
        cv2.putText(image, label, (bbox[0], bbox[1] + 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.3, (255,255,255), 4)

    return image

In [ ]:
rgb_color = matplotlib.colors.to_rgb('darkblue')
rgb_color = tuple(value * 255 for value in rgb_color)
rgb_color

In [ ]:
# image_id = os.listdir(train_image_path)[INDEX]
# masks = get_masks_by_img_id(train_df, image_id)
# masks = masks.astype(np.uint8)


# path = os.path.join(train_image_path,image_id)
# img = cv2.imread(path)
# img =  cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# colors = ['maroon', 'darkblue', 'purple','teal']
# rgb_colors = [(56, 255, 255),(255, 70, 90),(48, 255, 99),(255, 255, 102)]
# labels = ['Fish', 'Flower', 'Gravel', 'Sugar']


# plt.figure(figsize=(15,10))
# for i,(mask,cmap,label)in enumerate(zip(masks,colormaps,mask_labels)):
#     mask = np.clip(mask,0,1)
#     mask = np.ma.masked_where(mask == 0, mask)
#     ax = plt.subplot(2,2, i+1)
#     plt.imshow(img)
#     plt.imshow(mask,alpha=0.7,cmap=cmap)
#     draw_label_on_mask(mask,label)
#     plt.axis("off")
# #cv2.imwrite(f"{label}.jpg",cv2.cvtColor(img, cv2.COLOR_RGB2BGR))

In [ ]:
image_id = os.listdir(train_image_path)[INDEX]
masks = get_masks_by_img_id(train_df, image_id)
masks = masks.astype(np.uint8)


path = os.path.join(train_image_path,image_id)
img = cv2.imread(path)
img =  cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

colors = ['maroon', 'darkblue', 'purple','teal']
rgb_colors = [(56, 255, 255),(255, 70, 90),(48, 255, 99),(255, 255, 102)]
labels = ['Fish', 'Flower', 'Gravel', 'Sugar']

img = show_bounding_boxes(img,masks,labels,rgb_colors)

dpi = 100
plt.figure(figsize=(img_width/dpi, img_height/dpi), dpi=dpi)
#plt.figure(figsize=(32,8))
plt.imshow(img)
plt.axis("off");
#plt.savefig(f"{work_dir}All_bbox.png",transparent=True,bbox_inches='tight', pad_inches=0)

In [ ]:
def show_img_with_masks(img,masks,comment=""):
    
    colormaps = ['PuRd_r', 'Blues_r', 'Purples_r','winter_r']
    mask_labels = ['Fish', 'Flower', 'Gravel', 'Sugar']
    
    fig, axes = plt.subplots(1,6,figsize=(36,4))
    axes = axes.ravel()
    
    if img.shape[-1]!=3:
        img_cmap = 'gray'
    else:
        img_cmap=None
        
    for ix,axis in enumerate(axes):
        ix = ix%6
        axis.imshow(img,cmap=img_cmap)
        axis.axis('off')
        if ix==0:
            axis.set_title("Main Image")
        elif ix==1:
            for i,(mask,cmap,label) in enumerate(zip(masks,colormaps,mask_labels)):
                mask = np.clip(mask,0,1)
                mask = np.ma.masked_where(mask == 0, mask)
                axis.imshow(mask,alpha=0.7,cmap=cmap)
                axis.set_title(f"All the mask {comment}")
                draw_label_on_mask(mask,label,axis)
        elif ix>=2:
            for i,(mask,cmap,label) in enumerate(zip(masks,colormaps,mask_labels)):
                mask = np.clip(mask,0,1)
                mask = np.ma.masked_where(mask == 0, mask)
                axis = axes[2+i]
                axis.imshow(mask,alpha=0.4,cmap=cmap)
                axis.set_title(f"{label} {comment}")
                draw_label_on_mask(mask,label,axis)
    plt.show()
    
    return None

In [ ]:
image_id = os.listdir(train_image_path)[INDEX]
path = os.path.join(train_image_path,image_id)
img = cv2.imread(path)
img =  cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
masks = get_masks_by_img_id(train_df, image_id)
masks = (masks[:,:,0], masks[:,:,1],masks[:,:,2],masks[:,:,3])
show_img_with_masks(img,masks)

In [ ]:
image_ids = os.listdir(train_image_path)[13:16]
for image_id in image_ids:
    path = os.path.join(train_image_path,image_id)
    img = cv2.imread(path)
    img =  cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    masks = get_masks_by_img_id(train_df, image_id)
    masks = (masks[:,:,0], masks[:,:,1],masks[:,:,2],masks[:,:,3])
    show_img_with_masks(img,masks)